<a href="https://colab.research.google.com/github/pradeep-84/pinnacle-tasks/blob/main/Teachable_Machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import numpy as np
from PIL import Image

# ==========================================
# 1. DIAGNOSTICS & SETUP
# ==========================================
try:
    import gradio as gr
except ImportError:
    print("\n❌ ERROR: 'gradio' is not installed.")
    print("👉 Fix it by running this in your terminal: pip install gradio\n")
    sys.exit()

try:
    from sklearn.linear_model import LogisticRegression
except ImportError:
    print("\n❌ ERROR: 'scikit-learn' is not installed.")
    print("👉 Fix it by running this in your terminal: pip install scikit-learn\n")
    sys.exit()

# ==========================================
# 2. TEACHABLE MACHINE CORE LOGIC
# ==========================================

# Global data stores to hold our trained model state
trained_model = None
class_names = ["Class A", "Class B"]

def preprocess_image(pil_image):
    """Resizes and flattens an image into a raw numerical feature vector."""
    if pil_image is None:
        return None
    # Resize to a standard low-res layout to keep it incredibly fast without losing basic color/shape data
    img_resized = pil_image.resize((32, 32)).convert("RGB")
    # Convert pixels to a flat 1D numpy array of 3072 values (32 * 32 * 3 channels)
    return np.array(img_resized).flatten() / 255.0

def train_and_predict(class_a_files, class_b_files, test_image, name_a, name_b):
    """Trains a fresh classifier on the uploaded files and tests it against the target image."""
    global trained_model, class_names

    # Update custom class labels
    label_a = name_a.strip() if name_a.strip() else "Class A"
    label_b = name_b.strip() if name_b.strip() else "Class B"
    class_names = [label_a, label_b]

    X_train = []
    y_train = []

    # Process Class A examples
    if class_a_files:
        for file_path in class_a_files:
            img = Image.open(file_path)
            features = preprocess_image(img)
            X_train.append(features)
            y_train.append(0) # 0 represents Class A

    # Process Class B examples
    if class_b_files:
        for file_path in class_b_files:
            img = Image.open(file_path)
            features = preprocess_image(img)
            X_train.append(features)
            y_train.append(1) # 1 represents Class B

    # Validation Checks
    if len(X_train) == 0:
        return "❌ Error: Please upload sample images for your classes first!"
    if len(set(y_train)) < 2:
        return f"❌ Error: You need at least 1 image in BOTH '{label_a}' and '{label_b}' to train."
    if test_image is None:
        return "⏳ Model trained successfully! Now upload a 'Test Image' below to see it classify."

    # Train a standard Logistic Regression model on our image features instantly
    try:
        trained_model = LogisticRegression(max_iter=1000)
        trained_model.fit(np.array(X_train), np.array(y_train))

        # Predict the test image
        test_features = preprocess_image(test_image).reshape(1, -1)
        prediction_idx = trained_model.predict(test_features)[0]
        probabilities = trained_model.predict_proba(test_features)[0]

        confidence = probabilities[prediction_idx] * 100
        predicted_label = class_names[prediction_idx]

        return f"🔮 Prediction: {predicted_label}\n📊 Confidence: {confidence:.2f}%"

    except Exception as e:
        return f"An error occurred during training: {e}"

# ==========================================
# 3. WEB APP INTERFACE (GRADIO)
# ==========================================
print("🚀 Starting your local Teachable Machine app...")

with gr.Blocks() as demo:
    gr.Markdown("# 🤖 DIY Teachable Machine")
    gr.Markdown("Train your own custom AI model using image samples directly in your browser instantly!")

    with gr.Row():
        with gr.Column():
            name_a = gr.Textbox(value="Class A", label="Name for Class A (e.g., Apple)")
            class_a_input = gr.File(file_count="multiple", file_types=["image"], label="Upload Class A Training Images")

        with gr.Column():
            name_b = gr.Textbox(value="Class B", label="Name for Class B (e.g., Banana)")
            class_b_input = gr.File(file_count="multiple", file_types=["image"], label="Upload Class B Training Images")

    with gr.Row():
        with gr.Column():
            test_input = gr.Image(type="pil", label="Test Image (What is this?)")
            train_btn = gr.Button("⚡ Train Model & Classify", variant="primary")

        with gr.Column():
            output_text = gr.Textbox(label="Status & Prediction Output", lines=4)

    # Connect UI trigger
    train_btn.click(
        fn=train_and_predict,
        inputs=[class_a_input, class_b_input, test_input, name_a, name_b],
        outputs=output_text
    )

if __name__ == "__main__":
    demo.launch()

🚀 Starting your local Teachable Machine app...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5c60b9dc8c0fb1ae86.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
